# Support Vector Machine

In [1]:
import pandas as pd
import numpy as np
import cvxopt
from tqdm import tqdm

In [2]:
#Based on the 1995 paper by Vapnik and Cortes: https://www.marenglenbiba.net/dm/cortes_vapnik95.pdf
#Uses L2-loss instead

#Accuracy of ~96%

In [ ]:
def gaussian(x,y,sigma=2.0):
    return np.exp((-(np.linalg.norm(x-y, axis=1))**2)/(2*(sigma**2)))

def linear(x,y):
    return np.dot(x,y.T)

In [ ]:
class SVM():
    def __init__(self, kernel, C=0.5, max_sv=2500, threshold = -0.1):
        self.kernel = kernel
        self.C = C
        self.max_sv = max_sv  # Maximum number of support vectors to carry over
        self.sv = np.empty((0, 0))
        self.sv_y = np.empty((0,))
        self.sv_a = np.empty((0,))
        self.threshold_offset = threshold
        pass

    def batchRun(self, x, y, step=2000):
        for i in tqdm(range(0, x.shape[0], step), desc='Processing...'):
            batch_x = x[i:i+step]
            batch_y = y[i:i+step]

            if i > 0 and self.sv.shape[0] > 0:
                batch_x = np.vstack((batch_x, self.sv))
                batch_y = np.hstack((batch_y, self.sv_y))
                
            self.fit(batch_x, batch_y)
            self.getParameters()

    def fit(self, x, y):
        self.x = x
        self.y = y
        m, n = x.shape

        self.k = np.zeros((m, m))

        # Calculate the Kernel Matrix:
        for i in range(m):
            self.k[i, :] = self.kernel(x[i, np.newaxis], self.x) 

        d = cvxopt.matrix(np.outer(y, y) * self.k + np.eye(m) / self.C)
        i_vec = cvxopt.matrix(np.ones((m, 1)) * -1)
        G = cvxopt.matrix(np.eye(m) * -1)
        H = cvxopt.matrix(np.zeros((m, 1)))
        A = cvxopt.matrix(y, (1, m), 'd')
        b = cvxopt.matrix(0.0)

        cvxopt.solvers.options['show_progress'] = False
        sol = cvxopt.solvers.qp(d, i_vec, G, H, A, b)
        self.a = np.array(sol['x'])

    def getParameters(self):
        # L2-loss SVMs assign non-zero alpha to most points. 
        alpha_flat = self.a.flatten()
        num_candidates = min(len(alpha_flat), self.max_sv)
        
        top_indices = np.argsort(alpha_flat)[-num_candidates:]
        
        self.sv = self.x[top_indices]
        self.sv_y = self.y[top_indices]
        self.sv_a = alpha_flat[top_indices]

        b_sum = 0
        for k in range(len(self.sv)):
            kernel_dist = self.kernel(self.sv[k, np.newaxis], self.sv)
            b_sum += self.sv_y[k] - np.sum(self.sv_a * self.sv_y * kernel_dist)
        
        self.b = b_sum / len(self.sv) if len(self.sv) > 0 else 0.0

    def predict(self, x):
        y_pred = np.zeros((x.shape[0], 1))
        for i in range(x.shape[0]):
            y_pred[i] = np.sum(self.sv_a * self.sv_y * self.kernel(x[i, np.newaxis], self.sv))

        scores = y_pred + self.b
        return np.where(scores > self.threshold_offset, 1, -1).flatten()

In [5]:
def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))
def precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == -1)) 
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0
def recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == -1) & (y_true == 1)) 
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0
def f1(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

In [6]:
# Load test data 
test_df = pd.read_csv('data/data_encoded_test.csv').drop_duplicates()
X_test = test_df.drop(columns=['is_fraud']).values
y_test = test_df['is_fraud'].replace({0: -1}).values

df = pd.read_csv('data/data_encoded_train.csv')
X_train = df.drop(columns=['is_fraud']).values
y_train = df['is_fraud'].replace({0: -1}).values
svm = SVM(kernel= gaussian)
svm.batchRun(X_train,y_train)
y_pred = svm.predict(X_test)

print("Final Metrics:")
print(f'accuracy: {accuracy(y_test,y_pred)}')
print(f'recall: {recall(y_test,y_pred)}')
print(f'f1 score: {f1(y_test,y_pred)}')
print(f'precision: {precision(y_test,y_pred)}')

Processing...: 100%|██████████| 15/15 [05:14<00:00, 20.95s/it]


Final Metrics:
accuracy: 0.8992
recall: 0.8333333333333334
f1 score: 0.056179775280898875
precision: 0.029069767441860465
